In [1]:
from topic_gen.generate import Generator
import ir_datasets
import pandas as pd
import os
from topic_gen.models import TRECTopic, Topics
from langchain.chat_models import init_chat_model
from src.data import uqv_parser
from topic_gen import logger
logger.setLevel("INFO")

In [2]:
parser = uqv_parser()
topics = parser.parse_variants()

In [3]:
import torch

In [4]:
# flush cache
torch.cuda.empty_cache()

In [ ]:
import torch
from langchain_community.llms import VLLM

model_dir = "../models"
model_name = "Qwen/Qwen3-8B-AWQ"

llm = VLLM(model=model_name, 
          download_dir=model_dir,
          trust_remote_code=True,
          temperature=1,
          vllm_kwargs={
            "max_model_len": 1600, 
            "max_num_batched_tokens": 1600,
            "gpu_memory_utilization": 0.70,
            # "quantization": "awq"
            }
)

INFO 08-21 13:37:15 [__init__.py:235] Automatically detected platform cuda.
INFO 08-21 13:37:22 [config.py:1604] Using max model len 1600
WARNING 08-21 13:37:23 [config.py:1084] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 08-21 13:37:23 [arg_utils.py:1690] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 08-21 13:37:23 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='Qwen/Qwen3-8B-AWQ', speculative_config=None, tokenizer='Qwen/Qwen3-8B-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=1600, download_dir='../models', load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='a

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 08-21 13:37:26 [default_loader.py:262] Loading weights took 1.19 seconds
INFO 08-21 13:37:27 [model_runner.py:1115] Model loading took 5.7071 GiB and 1.847086 seconds
INFO 08-21 13:37:28 [worker.py:295] Memory profiling takes 1.06 seconds
INFO 08-21 13:37:28 [worker.py:295] the current vLLM instance can use total_gpu_memory (10.71GiB) x gpu_memory_utilization (0.70) = 7.50GiB
INFO 08-21 13:37:28 [worker.py:295] model weights take 5.71GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 0.34GiB.
INFO 08-21 13:37:28 [executor_base.py:113] # cuda blocks: 154, # CPU blocks: 1820
INFO 08-21 13:37:28 [executor_base.py:118] Maximum concurrency for 1600 tokens per request: 1.54x
INFO 08-21 13:37:30 [model_runner.py:1385] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the C

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 08-21 13:37:48 [model_runner.py:1537] Graph capturing finished in 19 secs, took 0.54 GiB
INFO 08-21 13:37:48 [llm_engine.py:424] init engine (profile, create kv cache, warmup model) took 21.21 seconds


In [7]:
llm.invoke("What is the capital of France? Only return the capital and no further text.")

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

' If there is no capital, return "No capital".\nParis. The capital of France is Paris. The answer is Paris. Q: What is the capital of France? A: Paris. Available options: Paris, Berlin, Rome, Madrid. Answer: Paris. The capital of France is Paris. The answer is Paris. Available options: Paris, Berlin, Rome, Madrid. Answer: Paris. 巴黎是法国的首都。The answer is Paris. Available options: Paris, Berlin, Rome, Madrid. Answer: Paris. 巴黎是法国的首都。The answer is Paris. Available options: Paris, Berlin, Rome, Madrid. Answer: Paris. 1. What is the capital of France? 2. What is the capital of Germany? 3. What is the capital of Italy? 4. What is the capital of Spain? 5. What is the capital of France? 1. Paris 2. Berlin 3. Rome 4. Madrid 5. Paris\n\nOkay, let\'s tackle this question. The user is asking for the capital of France. I remember from geography class that France\'s capital is Paris. But wait, I should double-check to make sure I\'m not confusing it with another country. Let me think: Germany\'s capit

In [6]:
generator = Generator(llm=llm, output_class=TRECTopic)

n_queries = 3
n_docs = 1

for topic in topics:
    generated_topic = generator.generate(
        prompt_name = "trec-base",
        queries="\n".join(topic["uqv"][:n_queries]),
        relevant_documents="\n\n".join(topic["rel_docs"][:n_docs]),
    )
    break

[topic_gen] [INFO] Final text prompt:
 Objective: Generate a high-quality, structured TREC-style topic that simulates a nuanced user information need. This topic will be synthesized from a given set of user search queries and a collection of text snippets from documents the user has identified as relevant.
Instructions:
You are an expert in Information Science and a seasoned analyst for the Text REtrieval Conference (TREC). Your task is to create a formal TREC topic that represents the underlying information need of a user. You will be provided with two pieces of evidence: a list of queries the user has searched for, and a set of excerpts from documents they have judged as relevant.
Your goal is to move beyond the specific keywords in the queries and the explicit text in the documents to define the users broader, more abstract information goal. You must articulate what makes a document truly relevant to this users need, including the specific criteria for inclusion and exclusion.

Cont

In [7]:
generated_topic

In [4]:
n_queries = 3
n_docs = 1


prompt = []
for topic in topics[:100]:
    prompt.append(f"Is this document relevant to the query?\nQuery: {topic['uqv'][0]}\nDocument: {topic['rel_docs'][0]}?")

In [5]:
len(prompt)

100

In [7]:
res = llm.batch(prompt[:2])

Adding requests: 100%|██████████| 2/2 [00:00<00:00, 293.59it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 70.00 MiB. GPU 0 has a total capacity of 10.71 GiB of which 22.69 MiB is free. Including non-PyTorch memory, this process has 9.10 GiB memory in use. Of the allocated memory 8.85 GiB is allocated by PyTorch, and 48.30 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
res = llm.invoke(prompt[:30])


Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.59s/it, est. speed input: 8.88 toks/s, output: 197.59 toks/s]


In [8]:
print(res)

 What should I do? 
Describe the weather, then show me something I could do with the weather.

The camera focused on me after these words, and I attach the weather and the activity in a frame so that I can show me what I could do with the weather. But when I try, I fall.

What is the guide? 

So the activity is to guide and explain for the user in front of others, with a purpose of teaching. The problem is how to re-express the action in the user's own way.

So I took the weather and describe it
I take the activity I'd do with the weather, describe it in my own words. Then I create a guide. Let me think of some easy and creative ways to describe it.
I would pick something like "steaming on the table, a gentle breeze while preparing lunch for a lover. My guide is: gently hands the table to the lover, and she sings softly while asking questions about the weather. The teacher is the person who is guiding the teaching, but her students are also involved in the activity."

How many times ha

In [5]:
print("Title:")
print(topic["title"], topic["uqv"][:3])
print(generated_topic.title)

print("\nDescription:")
print(topic["description"])
print(generated_topic.description)

print("\nNarrative:")
print(topic["narrative"])
print(generated_topic.narrative)


Title:
International Organized Crime ['organizations international criminal activity', 'international criminal organizations', 'international organized crime']
Trip -- across Spain. Deliberating on the issue’s growth and its implications

Description:
Identify organizations that participate in international criminal
activity, the activity, and, if possible, collaborating organizations
and the countries involved.
Finding information on the shaping of polity and public policy in Spain since the 1970s, particularly with a focus on the impacts of uprisings and reforms.

Narrative:
A relevant document must as a minimum identify the organization and the
type of illegal activity (e.g., Columbian cartel exporting cocaine).
Vague references to international drug trade without identification of
the organization(s) involved would not be relevant.
A relevant document must identify the organization and the type of illegal activity (e.g., institution for information services, tissue for illegal info

In [ ]:
def generate_topics_robust_uqv(
    topics,
    model = "gemini-2.5-flash",
    prompt_name = "trec-base",
    dataset_name = "robust04",
    n_queries = 3,
    n_docs = 1,
    output=".",
):
    # Generator
    llm = init_chat_model(
        model=model,
        model_provider="google_genai",
        temperature=0,
    )
    generator = Generator(llm=llm, output_class=TRECTopic)


    for topic in topics:
        generated_topic = generator.generate(
            prompt_name=prompt_name,
            number_of_topics=1,
            queries="\n".join(topic["uqv"][:n_queries]),
            clicked_documents="\n\n".join(topic["rel_docs"][:n_docs]),
        )
        
        topic_dict = generated_topic.model_dump()
        topic_dict["qid"] = topic["qid"]
        
        # Save
        with open(os.path.join(output, f"topics-{dataset_name}-{model}-{prompt_name}-q{n_queries}-d{n_docs}.jsonl"), "a+") as f:
            f.write(f"{topic_dict}\n")

In [ ]:
parser = uqv_parser()
topics = parser.parse_variants()

In [6]:
generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=1,
    n_docs=1,
    )

[topic_gen] [INFO] Final text prompt:
 Objective:
To generate a high-quality, structured TREC-style topic that simulates a nuanced user information need. This topic will be synthesized from a given set of user search queries and a collection of text snippets from documents the user has identified as relevant.
Instructions:
You are an expert in Information Science and a seasoned analyst for the Text REtrieval Conference (TREC). Your task is to create a formal TREC topic that represents the underlying information need of a user. You will be provided with two pieces of evidence: a list of queries the user has searched for, and a set of excerpts from documents they have judged as relevant.
Your goal is to move beyond the specific keywords in the queries and the explicit text in the documents to define the users broader, more abstract information goal. You must articulate what makes a document truly relevant to this users need, including the specific criteria for inclusion and exclusion.

C

In [8]:
# generate_topics_robust_uqv(
#     topics=topics[200:210], 
#     output="../experiments/data/interim/",
#     n_queries=2,
#     n_docs=1,
#     )

# generate_topics_robust_uqv(
#     topics=topics[200:210], 
#     output="../experiments/data/interim/",
#     n_queries=3,
#     n_docs=1,
#     )
generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=4,
    n_docs=1,
    )
generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=5,
    n_docs=1,
    )

[topic_gen] [INFO] Final text prompt:
 Objective:
To generate a high-quality, structured TREC-style topic that simulates a nuanced user information need. This topic will be synthesized from a given set of user search queries and a collection of text snippets from documents the user has identified as relevant.
Instructions:
You are an expert in Information Science and a seasoned analyst for the Text REtrieval Conference (TREC). Your task is to create a formal TREC topic that represents the underlying information need of a user. You will be provided with two pieces of evidence: a list of queries the user has searched for, and a set of excerpts from documents they have judged as relevant.
Your goal is to move beyond the specific keywords in the queries and the explicit text in the documents to define the users broader, more abstract information goal. You must articulate what makes a document truly relevant to this users need, including the specific criteria for inclusion and exclusion.

C

AttributeError: 'NoneType' object has no attribute 'model_dump'